# 03_04 On your own: a ticket router for Kittiwake Mobile

**The brief.** Kittiwake's support inbox sends every ticket to one of four teams: `billing`, `network`,
`device` or `account`. Today a person reads each one and forwards it. Build the model that does it.

- Train on `data/kittiwake_tickets.csv`: 600 tickets, each with its `department`.
- Choose a model by **cross-validated macro F1** on those 600 (macro F1 averages the F1 of the four
  departments, so a small team counts as much as a big one). Try at least two models.
- Predict the department of each of the 200 tickets in `data/kittiwake_tickets_unlabeled.csv`.
- Write `out/ticket_predictions.csv` with the columns `ticket_id,department`, one row per ticket, and
  `out/router_report.json` with `"model"` (a short description of what you chose) and `"cv_macro_f1"`.

**What you have.** Everything from this lab and the two before it: tokenizers, TF-IDF, pipelines,
logistic regression, SVMs, cross-validation. About a fifth of the tickets arrived by SMS, lowercased and
without apostrophes. Some tickets use another team's words on purpose; a ticket about being charged for a
returned phone belongs to billing.

**Tools you may use, any of them.** TF-IDF pipelines from this lab. A pipeline that starts with
`Embed()` from `spamtools.py` instead, so it reads meaning. Or no training at all:
`describe_classes(teams, texts)` routes each text to the team whose one-sentence description is nearest, as
on the chapter's first page. The embeddings of all 800 tickets were prepared in the background when your
session started; if that has not finished, the first run computes them, up to about a minute and a half.

**How it is checked.** `check_on_your_own()` checks the format and your report. The checkpoint scores your
200 predictions against the departments Kittiwake's team assigned, which this notebook never sees, and asks
for a macro F1 of 0.85. After you save, the last section looks at what your errors are made of.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-03-teaching-a-machine-what-spam-looks-like", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'nltk': 'nltk',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib',
           'joblib': 'joblib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import pandas as pd
from sklearn.model_selection import cross_val_score
from nlpcheck import check_on_your_own

tickets = pd.read_csv("data/kittiwake_tickets.csv")
unlabeled = pd.read_csv("data/kittiwake_tickets_unlabeled.csv")
print(tickets["department"].value_counts())
tickets.head()

In [ ]:
# YOUR CODE HERE: build candidate models, compare them with cross_val_score(..., scoring="f1_macro"),
# fit the best on all 600 tickets, and predict the 200.
predictions = ["billing"] * len(unlabeled)
report = {"model": "", "cv_macro_f1": None}

In [ ]:
os.makedirs("out", exist_ok=True)
pd.DataFrame({"ticket_id": unlabeled["ticket_id"], "department": predictions}).to_csv(
    "out/ticket_predictions.csv", index=False)
json.dump(report, open("out/router_report.json", "w"), indent=1)
check_on_your_own()

## After you save: what are the errors made of?

Your cross-validated score is not 1.0, and it is worth knowing why before you try to push it higher. The
cells below use one fixed model, TF-IDF and logistic regression, so that everyone sees the same numbers.
`cross_val_predict` gives every one of the 600 labelled tickets a prediction from a model that never saw
it, so you can compare prediction and label for all of them.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from nlpcheck import guess, reveal, misrouted_among

fixed = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
cv_pred = cross_val_predict(fixed, tickets["text"], tickets["department"], cv=5)
teams = ["account", "billing", "device", "network"]
print(pd.DataFrame(metrics.confusion_matrix(tickets["department"], cv_pred, labels=teams),
                   index=["really " + t for t in teams], columns=["said " + t for t in teams]))
errors = tickets[cv_pred != tickets["department"]].assign(model_said=cv_pred[cv_pred != tickets["department"]])
print(len(errors), "tickets where the model and the label disagree")

Read the disagreements before you blame the model. Each row shows the label Kittiwake's triage team gave
the ticket and what the model said instead.

In [ ]:
pd.set_option("display.max_colwidth", 110)
errors[["ticket_id", "department", "model_said", "text"]]

Predict: of these disagreements, in how many is the **model** right and the **label** wrong, because the
triage team forwarded the ticket to the wrong team? Read a dozen of them, then commit to a number.

In [ ]:
guess("label_wrong", None)   # a number

In [ ]:
reveal("label_wrong", misrouted_among(errors["ticket_id"]))

31 of the 35. T10080, "there is a charge of $73.25 i dont recognise on my last invoice", labelled `network`;
T10002, "I want to move my Aster Fold contract to my daughter's account", labelled `device`: the words say
one team and the label says another, because a person forwarded it in a hurry. About one ticket in twenty in this inbox was misrouted, and the model,
trained on all of them, still routes by what the ticket says. Only 4 of the 35 are genuine model mistakes,
mostly account tickets that mention restarting the phone.

Two consequences. **The errors you can fix are few**: a model that scored 1.0 here would have learned to
copy the triage team's mistakes, which is not what Kittiwake wants. And **the score has a ceiling**: on the
200 tickets the checkpoint holds, a reader who routed every ticket perfectly from its text would still
disagree with the labels on 16 of them, a macro F1 of 0.921. This is **label noise**, and every real
labelled dataset has some. Before tuning a model, read its errors; sometimes the fix is in the labels.

When you are done, go back to the lab instructions for the sign-off and the checkpoint.